# Experimentos com Redes Neurais — Ablação de Atributos Clínicos

Este notebook avalia o impacto da remoção de preditores-chave (`ChestPainType` e `ExerciseAngina`) no desempenho de uma rede neural densa (MLP) para predição de doença cardíaca.

In [ ]:
# Instalação (se necessário no Colab)
# !pip install tensorflow -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense


In [ ]:
# Carregamento da base com suporte local ou Google Colab
caminhos = [Path("data/heart.csv"), Path("heart.csv"), Path("../data/heart.csv")]
caminho_encontrado = None
for p in caminhos:
    if p.exists():
        caminho_encontrado = p
        break

if caminho_encontrado:
    df = pd.read_csv(caminho_encontrado)
else:
    try:
        from google.colab import files
        print("Faça o upload do arquivo heart.csv:")
        uploaded = files.upload()
        df = pd.read_csv("heart.csv")
    except Exception:
        url = "https://raw.githubusercontent.com/jayachandru001/Heart-Failure-Prediction-/main/heart.csv"
        df = pd.read_csv(url)

print(f"Dimensões da base: {df.shape}")
df.head()


## Função de Experimento (Treinamento e Avaliação da Rede MLP)

Treina uma arquitetura MLP (32 neurônios -> 16 neurônios -> 1 neurônio Sigmoide) com pré-processamento via `ColumnTransformer` (OneHotEncoder para categóricas e StandardScaler para numéricas).

In [ ]:
def executar_experimento(nome_modelo, remover=[]):
    print("\n" + "="*70)
    print(nome_modelo)
    print("="*70)

    colunas_remover = ["HeartDisease"] + remover
    X = df.drop(columns=colunas_remover)
    y = df["HeartDisease"]

    categoricas = [
        "Sex",
        "ChestPainType",
        "RestingECG",
        "ExerciseAngina",
        "ST_Slope"
    ]

    numericas = [
        "Age",
        "RestingBP",
        "Cholesterol",
        "FastingBS",
        "MaxHR",
        "Oldpeak"
    ]

    categoricas = [c for c in categoricas if c not in remover]
    numericas = [c for c in numericas if c not in remover]

    preprocessador = ColumnTransformer([
        ("cat", OneHotEncoder(drop="first"), categoricas),
        ("num", StandardScaler(), numericas)
    ])

    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=0.20,
        random_state=42,
        stratify=y
    )

    X_train = preprocessador.fit_transform(X_train)
    X_test = preprocessador.transform(X_test)

    modelo = Sequential([
        Dense(32, activation='relu', input_shape=(X_train.shape[1],)),
        Dense(16, activation='relu'),
        Dense(1, activation='sigmoid')
    ])

    modelo.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    modelo.fit(
        X_train, y_train,
        epochs=100,
        batch_size=32,
        verbose=0
    )

    probabilidades = modelo.predict(X_test, verbose=0)
    predicoes = (probabilidades > 0.5).astype(int)

    acc = accuracy_score(y_test, predicoes)
    prec = precision_score(y_test, predicoes)
    rec = recall_score(y_test, predicoes)
    f1 = f1_score(y_test, predicoes)
    auc = roc_auc_score(y_test, probabilidades)

    print(f"Acurácia : {acc:.4f}")
    print(f"Precisão : {prec:.4f}")
    print(f"Recall   : {rec:.4f}")
    print(f"F1-Score : {f1:.4f}")
    print(f"ROC AUC  : {auc:.4f}")

    return {
        "Modelo": nome_modelo,
        "Acurácia": acc,
        "Precisão": prec,
        "Recall": rec,
        "F1": f1,
        "ROC_AUC": auc
    }


## Execução dos Experimentos de Ablação

- **M1**: Modelo Completo (todas as variáveis)
- **M2**: Sem `ChestPainType` (Tipo de dor torácica)
- **M3**: Sem `ExerciseAngina` (Angina induzida por esforço)
- **M4**: Sem `ChestPainType` e sem `ExerciseAngina`

In [ ]:
resultados = []

# M1 - MODELO COMPLETO
resultados.append(executar_experimento("M1 - MODELO COMPLETO", remover=[]))

# M2 - SEM ChestPainType
resultados.append(executar_experimento("M2 - SEM ChestPainType", remover=["ChestPainType"]))

# M3 - SEM ExerciseAngina
resultados.append(executar_experimento("M3 - SEM ExerciseAngina", remover=["ExerciseAngina"]))

# M4 - SEM ChestPainType + ExerciseAngina
resultados.append(executar_experimento("M4 - SEM ChestPainType + ExerciseAngina", remover=["ChestPainType", "ExerciseAngina"]))


## Comparação dos Resultados

In [ ]:
resultado_final = pd.DataFrame(resultados).sort_values(by="Acurácia", ascending=False)

print("\n" + "="*70)
print("RESULTADO FINAL")
print("="*70)
display(resultado_final)

# Gráfico comparativo
plt.figure(figsize=(10, 5))
sns.barplot(data=resultado_final, x="Modelo", y="Acurácia", palette="viridis")
plt.ylim(0.7, 1.0)
plt.title("Comparação dos Experimentos de Ablação")
plt.ylabel("Acurácia")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()
